# Lekcija 10 - AI agenti u produkciji

U ovoj lekciji ćete naučiti **produkcijske obrasce** za AI agente koristeći Microsoft Agent Framework (Python). Obuhvaćamo:

- **Promatranje** — dodavanje mjerenja vremena i zapisivanja interakcija agenata
- **Evaluacija** — korištenje evaluacijskog agenta za ocjenjivanje kvalitete odgovora
- **Upravljanje troškovima** — strategije za optimizaciju tokena i odabir modela

Scenarij je **putnički agent** koji pomaže korisnicima u planiranju putovanja, s nadzorom i evaluacijom kao slojevima iznad.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Razmatranja u proizvodnji

Premještanje AI agenata iz prototipova u proizvodnju zahtijeva pažljivu pozornost na tri stupa:

1. **Promatranje** — Potrebna vam je vidljivost u to što agent radi, koliko mu to traje i koje alate poziva. Bez praćenja i zapisivanja, otklanjanje pogrešaka u proizvodnji je gotovo nemoguće.

2. **Procjena** — Automatizirane kontrole kvalitete osiguravaju da odgovori agenta ostaju točni, potpuni i korisni tijekom vremena. Agent procjenitelj može ocjenjivati odgovore prema definiranim kriterijima.

3. **Upravljanje troškovima** — Korištenje tokena izravno utječe na troškove. Strategije poput optimizacije upita, izbora modela i spremanja rezultata pomažu u kontroli troškova bez žrtvovanja kvalitete.


## Izgradnja promatranog agenta

Definiramo alate za putovanje i omotavamo poziv agenta s vremenskim mjerenjem kako bismo mogli pratiti kašnjenje. U produkciji biste integrirali s OpenTelemetryjem ili sličnim sustavom za praćenje.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategije upravljanja troškovima

Kontrola troškova je ključna za proizvodne AI agente. Evo ključnih strategija:

| Strategija | Opis |
|---|---|
| **Optimizacija prompta** | Držite upute sustava sažetima. Uklonite redundantni kontekst kako biste smanjili ulazne tokene. |
    "| **Odabir modela** | Koristite manje, jeftinije modele (npr. GPT-5-mini) za jednostavne zadatke poput klasifikacije ili ekstrakcije, a veće modele rezervirajte za složene razloge. |\n",
| **Keširanje** | Keširajte rezultate alata i učestale upite kako biste izbjegli nepotrebne API pozive. |
| **Budžeti tokena** | Postavite limite `max_tokens` kako biste spriječili neočekivano duge odgovore. |
| **Grupiranje (Batching)** | Grupirajte više korisničkih upita u jedan API poziv gdje je to moguće. |

U praksi dobro funkcionira višeslojni pristup: usmjerite jednostavne zahtjeve brzom, jeftinom modelu i eskalirajte samo složene upite na sposobniji model.


## Sažetak

U ovoj lekciji naučili ste kako:

1. **Dodati promatranje** u interakcije agenata putem vremenskog praćenja i zapisivanja, postavljajući temelje za praćenje i nadzor.
2. **Automatski vrednovati odgovore agenata** koristeći evaluacijskog agenta koji ocjenjuje potpunost, točnost i korisnost.
3. **Upravljati troškovima** kroz optimizaciju uputa, odabir modela, predmemoriranje i upravljanje proračunom tokena.

Ovi proizvodni obrasci pomažu osigurati da vaši AI agenti budu pouzdani, mjerljivi i isplativi na velikoj skali.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
